In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_interview_fldt(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :

        # -------DATE_FILTER
        date_filter = f"businessdate = {run_date!r}"

        # -------SCHEMA
        # schema = (
        #     "InterviewId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "InterviewKey int,"
        #     "InterviewType string,"
        #     "Applicationkey int,"
        #     "CDI string,"
        #     "Code string,"
        #     "Question string,"
        #     "Answer string,"
        #     "MaritalStatus int,"
        #     "NumberOfDependentAdults int,"
        #     "NumberOfDependentChildren int,"
        #     "DeclaredIncome 	decimal(18,6),"
        #     "InformativeExpenses 	decimal(18,6),"
        #     "DeclaredExpenses 	decimal(18,6),"
        #     "StatedIncome 	decimal(18,6),"
        #     "HasNewCollateral boolean,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        # -------CHOOSE PATH
        curated_table_path = f"{curated_zone}.Interview"

        # -------CREATION OF DELTA TABLE
        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "fldt",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # --------APPLY FILTERS AND SELECT COLUMNS FROM THE TABLES WE NEED

        # ================== FINANCIAL_ASSESSMENT_INTERVIEW ==================

        financialassessmentinterview_extra_filters = None

        if financialassessmentinterview_extra_filters is not None:
            financialassessmentinterview_extra_filters.append(date_filter)
            financialassessmentinterview_final_filters = " AND ".join(
                financialassessmentinterview_extra_filters
            )
        else:
            financialassessmentinterview_final_filters = date_filter

        financialassessmentinterview_selected_fields = [
            "id",
            "MaritalStatus",
            "NumberOfDependentAdults",
            "NumberOfDependentChildren",
            "DeclaredIncome",
            "InformativeExpenses",
            "DeclaredExpenses",
            "StatedIncome",
            "HasNewCollateral",
            "MasterInterviewId",
            "BusinessDate",
        ]

        FINANCIAL_ASSESSMENT_INTERVIEW = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_financialassessmentinterview",
            financialassessmentinterview_final_filters,
            financialassessmentinterview_selected_fields,
        )

        # ================== APPLICATION ==================

        application_extra_filters = None

        if application_extra_filters is not None:
            application_extra_filters.append(date_filter)
            application_final_filters = " AND ".join(application_extra_filters)
        else:
            application_final_filters = date_filter

        application_selected_fields = [
            "ApplicationKey",
            "CustomerCode",
            "id",
        ]

        APPLICATION = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_application",
            application_final_filters,
            application_selected_fields,
        )

        # ================== MASTER_INTERVIEW ==================

        masterinterview_extra_filters = None

        if masterinterview_extra_filters is not None:
            masterinterview_extra_filters.append(date_filter)
            masterinterview_final_filters = " AND ".join(masterinterview_extra_filters)
        else:
            masterinterview_final_filters = date_filter

        masterinterview_selected_fields = [
            "Id",
            "CustomerId",
        ]

        MASTER_INTERVIEW = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_masterinterview",
            masterinterview_final_filters,
            masterinterview_selected_fields,
        )

        # ================== CUSTOMER ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "id",
            "ApplicationId",
            "CustomerCode",
        ]

        CUSTOMER = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ================== GENERAL_INTERVIEW ==================

        general_interview_extra_filters = None

        if general_interview_extra_filters is not None:
            general_interview_extra_filters.append(date_filter)
            general_interview_final_filters = " AND ".join(
                general_interview_extra_filters
            )
        else:
            general_interview_final_filters = date_filter

        general_interview_selected_fields = [
            "MasterInterviewId",
            "id",
            "Code",
            "Question",
            "Answer",
            "BusinessDate",
        ]

        GENERAL_INTERVIEW = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_generalInterview",
            general_interview_final_filters,
            general_interview_selected_fields,
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
            "Prefix",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # -------SET THE CONDITIONS FOR JOINS
        FinancialAssessmentInterview_MasterInterview_join_condition = (
            FINANCIAL_ASSESSMENT_INTERVIEW["MasterInterviewId"]
            == MASTER_INTERVIEW["Id"]
        )

        MasterInterview_Customer_join_condition = (
            MASTER_INTERVIEW["CustomerId"] == CUSTOMER["id"]
        )

        Application_Customer_join_condition = (
            APPLICATION["id"] == CUSTOMER["ApplicationId"]
        )

        General_Interview_Master_Interview_join_condition = (
            GENERAL_INTERVIEW["MasterInterviewId"] == MASTER_INTERVIEW["Id"]
        )

        Portfolio_Customer_join_condition = portfolio["Prefix"] == substring(
            CUSTOMER["CustomerCode"], 1, 3
        )

        # -------APPLY THE JOINS
        interview_df_1 = (
            FINANCIAL_ASSESSMENT_INTERVIEW.join(
                MASTER_INTERVIEW,
                FinancialAssessmentInterview_MasterInterview_join_condition,
                "inner",
            )
            .join(
                CUSTOMER,
                MasterInterview_Customer_join_condition,
                "inner",
            )
            .join(
                APPLICATION,
                Application_Customer_join_condition,
                "inner",
            )
            .join(
                portfolio,
                Portfolio_Customer_join_condition,
                "left",
            )
        )

        # --------SET THE KEYS FOR HASHKEY BASED ON DATA MAPPING
        interview_df_1_primary_key_fields = [
            FINANCIAL_ASSESSMENT_INTERVIEW.id,
            lit("fldt"),
        ]

        # -------CREATION OF HΑSKEY
        interview_df_1 = interview_df_1.withColumn(
            "InterviewId",
            sha2(
                concat_ws(";", *interview_df_1_primary_key_fields),
                256,
            ),
        ).withColumn(
            "InterviewKey",
            (
                concat_ws(
                    ";",
                    FINANCIAL_ASSESSMENT_INTERVIEW.id,
                    lit("FinancialAssessmentInterview"),
                )
            ),
        )
        # -------SELECTION OF FINAL COLUMNS
        Result_df_fldt = interview_df_1.select(
            interview_df_1.InterviewId.cast("string").alias("InterviewId"),
            lit("fldt").cast("string").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias(
                "PortfolioKey"
            ),  # APPLICATION.PortfolioCode.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias(
                "PortfolioName"
            ),  # APPLICATION.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            interview_df_1.InterviewKey.cast("int").alias("InterviewKey"),
            lit("FinancialAssessmentInterview").alias("InterviewType"),
            APPLICATION.ApplicationKey.cast("int").alias("Applicationkey"),
            APPLICATION.CustomerCode.cast("string").alias("CDI"),
            lit(None).alias("Code"),
            lit(None).alias("Question"),
            lit(None).alias("Answer"),
            FINANCIAL_ASSESSMENT_INTERVIEW.MaritalStatus.cast("int").alias(
                "MaritalStatus"
            ),
            FINANCIAL_ASSESSMENT_INTERVIEW.NumberOfDependentAdults.cast("int").alias(
                "NumberOfDependentAdults"
            ),
            FINANCIAL_ASSESSMENT_INTERVIEW.NumberOfDependentChildren.cast("int").alias(
                "NumberOfDependentChildren"
            ),
            FINANCIAL_ASSESSMENT_INTERVIEW.DeclaredIncome.cast("decimal(18,6)").alias(
                "DeclaredIncome"
            ),
            FINANCIAL_ASSESSMENT_INTERVIEW.InformativeExpenses.cast(
                "decimal(18,6)"
            ).alias("InformativeExpenses"),
            FINANCIAL_ASSESSMENT_INTERVIEW.DeclaredExpenses.cast("decimal(18,6)").alias(
                "DeclaredExpenses"
            ),
            FINANCIAL_ASSESSMENT_INTERVIEW.StatedIncome.cast("decimal(18,6)").alias(
                "StatedIncome"
            ),
            FINANCIAL_ASSESSMENT_INTERVIEW.HasNewCollateral.cast("boolean").alias(
                "HasNewCollateral"
            ),
            FINANCIAL_ASSESSMENT_INTERVIEW.BusinessDate.cast("date").alias(
                "BusinessDate"
            ),
        )

        # -------APPLY THE JOINS FOR THE SECOND ONE
        interview_df_2 = (
            GENERAL_INTERVIEW.join(
                MASTER_INTERVIEW,
                General_Interview_Master_Interview_join_condition,
                "inner",
            )
            .join(
                CUSTOMER,
                MasterInterview_Customer_join_condition,
                "inner",
            )
            .join(
                APPLICATION,
                Application_Customer_join_condition,
                "inner",
            )
            .join(
                portfolio,
                Portfolio_Customer_join_condition,
                "left",
            )
        )

        # --------SET THE KEYS FOR HASHKEY BASED ON DATA MAPPING FOR THE SECOND ONE
        interview_df_2_primary_key_fields = [
            GENERAL_INTERVIEW.id,
            lit("fldt"),
        ]

        # -------CREATION OF HΑSKEY FOR THE SECOND ONE
        interview_df_2 = interview_df_2.withColumn(
            "InterviewId",
            sha2(
                concat_ws(";", *interview_df_2_primary_key_fields),
                256,
            ),
        ).withColumn(
            "InterviewKey",
            (concat_ws(";", GENERAL_INTERVIEW.id, lit("GeneralInterview"))),
        )
        # -------SELECTION OF FINAL COLUMNS FOR THE SECOND ONE
        Result_df_fldt_2 = interview_df_2.select(
            interview_df_2.InterviewId.cast("string").alias("InterviewId"),
            lit("fldt").cast("string").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias(
                "PortfolioKey"
            ),  # APPLICATION.PortfolioCode.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias(
                "PortfolioName"
            ),  # APPLICATION.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            interview_df_2.InterviewKey.cast("int").alias("InterviewKey"),
            lit("GeneralInterview").alias("InterviewType"),
            APPLICATION.ApplicationKey.cast("int").alias("Applicationkey"),
            APPLICATION.CustomerCode.cast("string").alias("CDI"),
            GENERAL_INTERVIEW.Code.cast("string").alias("Code"),
            GENERAL_INTERVIEW.Question.cast("string").alias("Question"),
            GENERAL_INTERVIEW.Answer.cast("string").alias("Answer"),
            lit(None).alias("MaritalStatus"),
            lit(None).alias("NumberOfDependentAdults"),
            lit(None).alias("NumberOfDependentChildren"),
            lit(None).alias("DeclaredIncome"),
            lit(None).alias("InformativeExpenses"),
            lit(None).alias("DeclaredExpenses"),
            lit(None).alias("StatedIncome"),
            lit(None).alias("HasNewCollateral"),
            GENERAL_INTERVIEW.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================
        Result_df = Result_df_fldt_2.union(Result_df_fldt)

        Result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_interview_fldt(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)